In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, confusion_matrix

train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

X = train.drop(["readmitted_30_days", "ID"], axis=1)
y = train["readmitted_30_days"]
X_test = test.drop("ID", axis=1)

cat_cols = X.select_dtypes(include="object").columns
X = pd.get_dummies(X, columns=cat_cols, dtype=int)
X_test = pd.get_dummies(X_test, columns=cat_cols, dtype=int)
X_test = X_test.reindex(columns=X.columns, fill_value=0)

X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

model = LogisticRegression(penalty="l2", C=1.0)
model.fit(X_train, y_train)

y_prob = model.predict_proba(X_val)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)

print("ROC-AUC:", roc_auc_score(y_val, y_prob))
print(confusion_matrix(y_val, y_pred))

test_prob = model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "ID": test["ID"],
    "readmitted_30_days": test_prob
})
submission.to_csv("predictions.csv", index=False)
